# 00 Preflight Check

Run this notebook before each live session. It checks the teaching environment, required packages, course notebooks, and optional cloud credentials before participants join.

## How to Use

1. Select the `Python (AI Beginner Bootcamp)` kernel if it is available.
2. Choose **Restart Kernel and Run All Cells**.
3. Treat any `FAIL` result as a blocker for live delivery.
4. Treat `WARN` as optional setup that is only needed if you plan to demo that service.

In [ ]:
import importlib
import json
import os
import platform
import sys
from pathlib import Path

CHECKS = []

def record(name, status, detail=""):
    CHECKS.append({"name": name, "status": status, "detail": detail})
    print(f"{status:4} | {name} | {detail}")

def ok(name, detail=""):
    record(name, "PASS", detail)

def warn(name, detail=""):
    record(name, "WARN", detail)

def fail(name, detail=""):
    record(name, "FAIL", detail)

candidate_roots = [
    Path(os.getenv("BOOTCAMP_COURSE_DIR", "")) if os.getenv("BOOTCAMP_COURSE_DIR") else None,
    Path.cwd(),
    Path.cwd() / "ai-beginner-bootcamp-outline",
    Path.home() / "ai-beginner-bootcamp-outline",
]
ROOT = next(
    (path for path in candidate_roots if path and (path / "Module_00_Intro_to_AI.ipynb").exists()),
    Path.cwd(),
)
print(f"Course directory: {ROOT}")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")

## 1. Required Package Check

In [ ]:
required_imports = {
    "boto3": "boto3",
    "google-genai": "google.genai",
    "ipywidgets": "ipywidgets",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "python-dotenv": "dotenv",
    "requests": "requests",
}

for package_name, import_name in required_imports.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "installed")
        ok(f"Package: {package_name}", str(version))
    except Exception as exc:
        fail(f"Package: {package_name}", f"Install with: python -m pip install -r requirements.txt ({exc})")

## 2. Course Notebook Integrity

In [ ]:
expected_notebooks = [
    "Module_00_Intro_to_AI.ipynb",
    "Module_01_AI_Productivity.ipynb",
    "Module_02_AI_Automation.ipynb",
    "Module_03_Build_with_AI.ipynb",
    "Module_04_AI_Opportunities.ipynb",
]

for notebook_name in expected_notebooks:
    path = ROOT / notebook_name
    if not path.exists():
        fail(f"Notebook exists: {notebook_name}", "Missing from course folder")
        continue
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        cells = data.get("cells", [])
        code_cells = [cell for cell in cells if cell.get("cell_type") == "code"]
        ok(f"Notebook valid: {notebook_name}", f"{len(cells)} cells, {len(code_cells)} code cells")
    except Exception as exc:
        fail(f"Notebook valid: {notebook_name}", str(exc))

## 3. Network Check

In [ ]:
try:
    import requests
    response = requests.get("https://aws.amazon.com", timeout=10)
    if response.status_code < 500:
        ok("Internet access", f"aws.amazon.com returned HTTP {response.status_code}")
    else:
        warn("Internet access", f"aws.amazon.com returned HTTP {response.status_code}")
except Exception as exc:
    warn("Internet access", f"External HTTP request failed: {exc}")

## 4. AWS Identity and S3 Check

In [ ]:
try:
    import boto3
    session = boto3.Session()
    region = session.region_name or os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
    if region:
        ok("AWS region", region)
    else:
        warn("AWS region", "Set AWS_REGION if using Bedrock or region-specific services")

    sts = session.client("sts", region_name=region) if region else session.client("sts")
    identity = sts.get_caller_identity()
    ok("AWS identity", identity.get("Arn", "Identity available"))
except Exception as exc:
    warn("AWS identity", f"Only required for S3/Bedrock demos: {exc}")

bucket = os.getenv("BOOTCAMP_S3_BUCKET")
if bucket:
    try:
        s3 = boto3.client("s3", region_name=region) if region else boto3.client("s3")
        s3.head_bucket(Bucket=bucket)
        ok("S3 bucket", bucket)
    except Exception as exc:
        fail("S3 bucket", f"Cannot access {bucket}: {exc}")
else:
    warn("S3 bucket", "BOOTCAMP_S3_BUCKET is not set; skip if not using S3 in the live demo")

## 5. Optional Gemini API Smoke Test

Only needed if Day 4 uses Google AI Studio live instead of the simulated response.

In [ ]:
gemini_key = os.getenv("GEMINI_API_KEY")
if not gemini_key:
    warn("Gemini API key", "GEMINI_API_KEY is not set; use the simulated fallback or set the key before Day 4")
else:
    try:
        from google import genai
        client = genai.Client(api_key=gemini_key)
        response = client.models.generate_content(
            model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
            contents="Reply with exactly: bootcamp ready",
        )
        text = getattr(response, "text", "")
        if "bootcamp ready" in text.lower():
            ok("Gemini smoke test", text.strip())
        else:
            warn("Gemini smoke test", f"Unexpected response: {text[:120]}")
    except Exception as exc:
        fail("Gemini smoke test", str(exc))

## 6. Optional Amazon Bedrock Smoke Test

Only needed if you choose the AWS-native model path for Day 4.

In [ ]:
bedrock_model_id = os.getenv("BEDROCK_MODEL_ID")
if not bedrock_model_id:
    warn("Bedrock model", "BEDROCK_MODEL_ID is not set; skip if not using Bedrock live")
else:
    try:
        import boto3
        bedrock_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or region
        client = boto3.client("bedrock-runtime", region_name=bedrock_region)
        ok("Bedrock client", f"Client created for {bedrock_model_id} in {bedrock_region}")
        warn("Bedrock invocation", "Client creation passed. Add a provider-specific test payload before a Bedrock live demo.")
    except Exception as exc:
        fail("Bedrock client", str(exc))

## 7. Final Readiness Summary

In [ ]:
from collections import Counter

counts = Counter(check["status"] for check in CHECKS)
print("Summary:")
for status in ["PASS", "WARN", "FAIL"]:
    print(f"{status}: {counts.get(status, 0)}")

if counts.get("FAIL", 0):
    print("\nResult: NOT READY. Fix FAIL items before the live session.")
elif counts.get("WARN", 0):
    print("\nResult: READY for local demos. Review WARN items if using optional cloud services.")
else:
    print("\nResult: READY. All checks passed.")